# 06_collect — 팀 결과 통합 점검과 보고서 표·그림 (담당: core)

`outputs/{logit, xgb, xgb_mono}/`에 올라온 결과를 읽어 두 가지를 합니다.

1. **점검:** 세 모형이 같은 버전(config·lib_prep 해시), 같은 열(contract), 같은 B, 같은 출력 형식으로 돌았는지 확인합니다. 하나라도 다르면 표를 만들기 전에 멈추고 무엇이 다른지 알려 줍니다.
2. **보고서용:** 표 4(`report/tables/t4_validation_summary.csv`), 그림 2·3·4(`report/figures/fig2~4`)를 만듭니다.

아직 올라오지 않은 모형은 "없음"으로 표시하고 있는 모형만으로 만듭니다. 그림 4(ΔSR)는 ②가 있어야 그립니다.
B=100 본 실행 결과가 모두 모이면 이 노트북만 다시 실행하면 됩니다.

In [1]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
%run -i ./lib_eval.ipynb
import json, joblib
import matplotlib.pyplot as plt
from matplotlib import font_manager
for name in ("AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
    if name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = name; break
plt.rcParams["axes.unicode_minus"] = False
MODELS = {"logit": "① 로지스틱", "xgb": "② XGBoost", "xgb_mono": "③ 단조 XGBoost"}
TAB_DIR = REPORT_DIR / "tables"; FIG_DIR = REPORT_DIR / "figures"
TAB_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
NOW = dict(config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH)
print("지금 버전:", NOW)

# 선택 파일 보호 (PR #4 리뷰): 실행을 시작하면 이전 selection.json을 먼저 무효화한다.
# 마지막 칸(6절)이 점검 통과 + 최종 결정일 때만 valid=true로 다시 쓰므로, 중간 어디서 멈춰도 이전 선택이 유효한 채 남지 않는다.
import hashlib, datetime, io
SEL_DIR = OUT_DIR / "collect"; SEL_PATH = SEL_DIR / "selection.json"
now_iso = lambda: datetime.datetime.now().isoformat(timespec="seconds")

def revoke_selection(reason):
    """이전 선택이 있으면 valid=false로 덮어써서 무효화한다."""
    if not SEL_PATH.exists():
        return
    try:
        prev = json.loads(SEL_PATH.read_text(encoding="utf-8"))
    except Exception:
        prev = {}
    keep = prev.get("previous") if prev.get("valid") is False else dict(selected_model=prev.get("selected_model"), created=prev.get("created"))
    SEL_PATH.write_text(json.dumps(dict(valid=False, selected_model=None, reason=reason, revoked=now_iso(), previous=keep),
                                   ensure_ascii=False, indent=1), encoding="utf-8")
    print("selection.json 무효화(valid=false):", reason)

def read_hashed(path):
    """파일을 한 번만 읽어 (바이트, sha256)을 돌려준다. 파싱과 지문 계산에 같은 바이트를 쓰기 위함."""
    b = Path(path).read_bytes()
    return b, hashlib.sha256(b).hexdigest()

revoke_selection("06_collect 실행 중: 최종 판정 전 (%s)" % now_iso())

ModuleNotFoundError: No module named 'nbformat'

In [ ]:
# 1. 점검 (모형마다 PASS/FAIL, 없는 모형은 건너뜀)
REQ_RES = ["b", "a_star", "tau_star", "sr_max", "auc", "brier"]
REQ_CURVE = ["b", "rate", "sr", "tau"]
REQ_SUMMARY = ["model", "a_star", "tau_star_median", "columns", "config_hash", "lib_prep_hash"]
cols_contract = {"logit": json.load(open(CONTRACT_DIR / "columns_logit.json")),
                 "xgb": json.load(open(CONTRACT_DIR / "columns_xgb.json"))}
cols_contract["xgb_mono"] = cols_contract["xgb"]

have, report = {}, []
for m, label in MODELS.items():
    d = OUT_DIR / m
    if not (d / "summary.json").exists():
        report.append((label, "없음", "outputs/%s/summary.json 없음" % m)); continue
    try:   # 파싱한 바이트로 지문을 계산 (6절이 이 지문을 selection.json에 기록하고, 저장 직전 디스크와 다시 대조)
        raw = {name: read_hashed(d / name) for name in ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv")}
        s = json.loads(raw["summary.json"][0].decode("utf-8"))
        res = pd.read_csv(io.BytesIO(raw["bootstrap_results.csv"][0])); cur = pd.read_csv(io.BytesIO(raw["bootstrap_curves.csv"][0]))
    except Exception as e:
        report.append((label, "FAIL", f"결과 파일 읽기 실패: {type(e).__name__}: {e}"[:200])); continue
    n_models = len(list((d / "models_b").glob("model_b*.joblib"))) if (d / "models_b").exists() else 0
    checks = {
        "summary 공통 키": all(k in s for k in REQ_SUMMARY),
        "config 해시 = 지금": s.get("config_hash") == NOW["config_hash"],
        "lib_prep 해시 = 지금": s.get("lib_prep_hash") == NOW["lib_prep_hash"],
        "열 = contract": s.get("columns") == cols_contract[m],
        "결과 열 형식": list(res.columns) == REQ_RES,
        "곡선 열 형식": list(cur.columns) == REQ_CURVE,
        "회차 수 = B": len(res) == B,
        "보관 모형 수 = 회차 수": n_models == len(res),
        "최종 모형 있음": (d / "final_model.joblib").exists(),
        "a* = 회차 중앙값": np.isclose(s.get("a_star", np.nan), res["a_star"].median()),
        # summary가 같은 실행에서 쓴 CSV 지문(files)과 지금 읽은 CSV가 같아야 함 (이전 summary + 새 CSV 섞임 방지)
        "CSV = summary.files": all((s.get("files") or {}).get(k) == raw[k][1] for k in ("bootstrap_results.csv", "bootstrap_curves.csv")),
    }
    bad = [k for k, v in checks.items() if not v]
    report.append((label, "PASS" if not bad else "FAIL", ", ".join(bad) if bad else f"B={len(res)}"))
    have[m] = dict(summary=s, res=res, curves=cur, inputs={f"{m}/{k}": h for k, (_, h) in raw.items()})
chk = pd.DataFrame(report, columns=["모형", "판정", "비고"])
print(chk.to_string(index=False))
FAILED = chk["판정"].eq("FAIL").any()
if FAILED:
    print("\nFAIL이 있으면 표와 그림은 '시험용'으로만 만듭니다. 담당자에게 비고의 항목을 고쳐 달라고 요청하세요.")

### 1-1. 분할 지문 비교 (4단계: 세 모형이 같은 분할)
각 모형 `summary.json`의 `split_hash`(B회 OOB 인덱스를 이어 붙인 sha256 앞 12자)가 모두 같아야 회차끼리 짝지어 뺄 수 있습니다. 다르거나 없으면 FAIL로 표시하고 표·그림을 `_시험용`으로 만듭니다.

In [ ]:
# 1-1. 분할 지문 비교
split = {m: have[m]["summary"].get("split_hash") for m in have}
n_rows = {m: have[m]["summary"].get("n_samples") for m in have}
for m, h in split.items():
    print(f"{MODELS[m]:<12} split_hash={h}  n_samples={n_rows[m]}")
missing = [MODELS[m] for m, h in split.items() if not h]
SPLIT_OK = bool(split) and not missing and len(set(split.values())) == 1
if missing:
    print("split_hash 없음 (split_hash 추가 전 실행):", ", ".join(missing), "→ 새 model_template·02_logit으로 다시 실행 필요")
elif len(set(split.values())) > 1:
    print("분할 지문이 서로 다름 → 같은 n·B·seed로 돌았는지 확인. 짝지은 비교를 믿을 수 없음")
else:
    print("분할 지문 일치:", next(iter(split.values())) if split else "-")
if not SPLIT_OK:
    FAILED = True

In [ ]:
# 2. 표 4: Dev OOB 요약 (회차 B개)
rows = []
for m, label in MODELS.items():
    if m not in have: continue
    r = have[m]["res"]
    rows.append({"모형": label, "B": len(r), "a* 중앙값": r["a_star"].median(),
                 "a* 범위": f"{r['a_star'].min():.2f}~{r['a_star'].max():.2f}",
                 "SR* 평균": r["sr_max"].mean(), "AUC 평균": r["auc"].mean(), "Brier 평균": r["brier"].mean()})
t4 = pd.DataFrame(rows).round(4)
tag = "_시험용" if FAILED else ""
t4.to_csv(TAB_DIR / f"t4_validation_summary{tag}.csv", index=False, encoding="utf-8-sig")
print(t4.to_string(index=False))

### 2-1. 승인율 a* = 평균 곡선의 최댓값 (5단계)
같은 승인율의 SR B개를 평균한 곡선에서 `MIN_RATE` 이상 구간의 최댓값 지점을 a*로 씁니다(기대 SR 최대화). 최댓값에서 0.005 이내 승인율 구간을 함께 적고, 회차별 최적값의 중앙값은 견고성 기록으로 둡니다.
결과: `report/tables/t4b_astar_mean_curve.csv`

In [ ]:
# 2-1. 평균 곡선 a*
NEAR_MAX = 0.005   # 아키텍처 5단계: 최댓값에서 0.005 이내 구간도 보고
A_STAR, rows = {}, []
for m, label in MODELS.items():
    if m not in have: continue
    cur = have[m]["curves"].assign(rate=lambda d: d["rate"].round(4))
    mean_c = cur[cur["rate"] >= MIN_RATE - 1e-9].groupby("rate")["sr"].mean().dropna()
    a_m, sr_m = float(mean_c.idxmax()), float(mean_c.max())
    near = mean_c[mean_c >= sr_m - NEAR_MAX].index
    A_STAR[m] = a_m
    rows.append({"모형": label, "a* (평균 곡선 최대)": a_m, "평균 곡선 SR 최대": sr_m,
                 "0.005 이내 구간": f"{near.min():.2f}~{near.max():.2f}",
                 "회차별 a*_b 중앙값 (견고성)": have[m]["res"]["a_star"].median(), "B": cur["b"].nunique()})
t4b = pd.DataFrame(rows).round(4)
t4b.to_csv(TAB_DIR / f"t4b_astar_mean_curve{tag}.csv", index=False, encoding="utf-8-sig")
print(t4b.to_string(index=False))

In [ ]:
# 3. 그림 2 (승인율별 SR 평균 ±1SD)와 그림 3 (회차별 a*_b 분포)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for i, (m, label) in enumerate(MODELS.items()):
    if m not in have: continue
    g = have[m]["curves"].groupby("rate")["sr"]; mu, sd = g.mean(), g.std(ddof=1).fillna(0)
    ax.plot(mu.index, mu.values, label=label, color=f"C{i}")
    ax.fill_between(mu.index, mu - sd, mu + sd, alpha=0.15, color=f"C{i}")
    ax.axvline(have[m]["res"]["a_star"].median(), ls="--", lw=0.8, color=f"C{i}")
ax.set_xlabel("승인율 a"); ax.set_ylabel("OOB SR (거절=0)"); ax.legend(frameon=False, fontsize=8)
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fig2_oob_sr_curves{tag}.png", dpi=300)

fig, ax = plt.subplots(figsize=(6.5, 2.8))
for i, (m, label) in enumerate(MODELS.items()):
    if m not in have: continue
    ax.hist(have[m]["res"]["a_star"], bins=np.arange(0.30, 0.71, 0.01), histtype="step", lw=1.6, label=label, color=f"C{i}")
ax.set_xlabel("회차별 최적 승인율 a*_b"); ax.set_ylabel("회차 수"); ax.legend(frameon=False, fontsize=8)
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fig3_astar_dist{tag}.png", dpi=300)
plt.show()

In [ ]:
# 4. 그림 4: 회차별 ΔSR (②의 a*_b 지점 기준, ② − 다른 모형). ②가 없으면 건너뜀
if "xgb" not in have:
    print("② 결과가 없어 그림 4는 건너뜀")
else:
    ref = have["xgb"]["curves"]; a_ref = have["xgb"]["res"].set_index("b")["a_star"]
    fig, ax = plt.subplots(figsize=(6.5, 3.2)); rows4 = []
    for i, m, label in ((2, "xgb_mono", "② − ③ (단조 제약 비용)"), (0, "logit", "② − ① (ML 이득)")):
        if m not in have: continue
        if m == "xgb_mono" and "DELTA" in PENDING:   # [미정] δ 확정 전에는 ②−③을 보지 않는다(판정 기준이 결과에 끌려가지 않게)
            print("δ 미정: ②−③ 곡선은 δ를 00_config에 확정한 뒤 다시 실행하면 그립니다"); continue
        mm = ref.merge(have[m]["curves"], on=["b", "rate"], suffixes=("_r", "_n"))
        mm["d"] = mm["sr_r"] - mm["sr_n"]
        mean = mm.groupby("rate")["d"].mean()
        ax.plot(mean.index, mean.values, label=label, color=f"C{i}")
        at = mm[np.isclose(mm["rate"], mm["b"].map(a_ref))]
        rows4.append({"비교": label, "ΔSR 중앙값(②의 a*_b)": at["d"].median(),
                      "최소": at["d"].min(), "최대": at["d"].max(), "음수 회차": int((at["d"] < 0).sum())})
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("승인율 a"); ax.set_ylabel("ΔSR (OOB, 회차 평균)")
    ax.legend(frameon=False, fontsize=8)
    for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
    fig.tight_layout(); fig.savefig(FIG_DIR / f"fig4_oob_dsr{tag}.png", dpi=300); plt.show()
    print(pd.DataFrame(rows4).round(4).to_string(index=False))

In [ ]:
# 5. 부록 C: ② 최종 모형의 PDP (부호 ±1 변수 20개, Dev 무작위 5,000건, 원 단위 5~95% 분위수 격자)
f = OUT_DIR / "xgb" / "final_model.joblib"
if not f.exists():
    print("② 최종 모형 없음 → 부록 C 건너뜀")
else:
    prep, model = joblib.load(f)
    Xs = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet").sample(5_000, random_state=SEED)
    cols = list(SIGN_OF); nr = int(np.ceil(len(cols) / 5))
    fig, axs = plt.subplots(nr, 5, figsize=(6.5, 1.5 * nr), squeeze=False); rows = []
    for ax, c in zip(axs.flat, cols):
        s_ = pdp_1d(prep, model, Xs, c)
        rho = pd.Series(s_.values).corr(pd.Series(s_.index), method="spearman") if len(s_) > 2 else np.nan
        ax.plot(s_.index, s_.values, color="C1" if SIGN_OF[c] > 0 else "C0", lw=1.2)
        ax.set_title(f"{c} ({SIGN_OF[c]:+d})", fontsize=5.5); ax.tick_params(labelsize=4.5); tidy(ax)
        rows.append(dict(변수=c, 부호=SIGN_OF[c], pdp_rho=rho, 변화폭=s_.iloc[-1] - s_.iloc[0]))
    for ax in list(axs.flat)[len(cols):]: ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / f"figA2_pdp{tag}.png", dpi=300); plt.show()
    pdt = pd.DataFrame(rows)
    pdt["판정"] = np.where(pdt["변화폭"].abs() < 1e-3, "평평(|변화폭|<0.001)", np.where(np.sign(pdt["변화폭"]) == pdt["부호"], "일치", "반대"))   # [미정·기본값] 평평 기준 0.001
    pdt.round(4).to_csv(TAB_DIR / f"tA3_pdp_check{tag}.csv", index=False, encoding="utf-8-sig")
    print(pdt.round(4).to_string(index=False))

### 6. 모델 선택: 짝지은 차이 (아키텍처 6단계) → `selection.json`
모형마다 자기 a*(2-1)에 고정한 회차별 SR을 구하고, 같은 회차끼리 빼서 차이 B개를 만듭니다.
- ②−①의 2.5% 분위수 > 0 이면 ML 이득
- ②−③의 97.5% 분위수 < δ 이면 ③ 선택, 아니면 ② (`ALPHA` = 0.025)
- OOB 구간은 정식 신뢰구간이 아니라 선택 근거입니다.

**공유 파일 보호 (PR #4 리뷰 1·2차 반영)**
- `outputs/collect/selection.json`은 **항상 마지막 06_collect 실행의 판정**만 담습니다.
  - 점검 통과 + 세 모형 + δ 확정 → `valid: true`와 선택 결과, 입력 파일 지문(`inputs`)을 씁니다.
  - 점검 실패(1절 FAIL, 분할 지문 불일치·누락) → 이전 선택이 있으면 **`valid: false`로 덮어써서 무효화**하고 멈춥니다. 선택은 계산하지 않습니다.
  - 모형 누락·δ 미정 → 이전 선택이 있으면 `valid: false`로 무효화하고, 새 선택은 쓰지 않습니다.
- `inputs`는 선택에 쓴 `outputs/{logit,xgb,xgb_mono}/` 의 `summary.json`·`bootstrap_results.csv`·`bootstrap_curves.csv` sha256입니다. 선택 뒤에 모형 결과가 바뀌면 지문이 달라져서 바로 드러납니다.
- **실행을 시작하면(첫 칸) 이전 `selection.json`을 먼저 `valid: false`로 바꿉니다.** 그래서 1절 CSV 읽기 실패 등 중간 어디서 멈춰도 이전 선택이 유효한 채 남지 않습니다.
- 모형 노트북은 결과 CSV를 쓴 뒤 마지막에 `summary.json`을 쓰고, 그 안에 **같은 실행의 CSV sha256(`files`)**을 남깁니다(PR #3). 1절은 읽은 CSV가 `summary.files`와 다르면 그 모형을 FAIL로 표시합니다. 모형 노트북이 CSV는 새로 쓰고 summary는 아직 이전 것인 순간에 실행해도 선택이 만들어지지 않습니다.
- `inputs` 지문은 1절에서 **파싱한 바로 그 바이트**로 계산하고, 저장 직전에 디스크 파일과 다시 대조합니다. 그 사이 Drive에 새 결과가 올라왔으면 저장하지 않고 멈춥니다.
- 선택 칸은 노트북 맨 끝에 있어서 여기서 멈춰도 표·그림에는 영향이 없습니다.

**`selection.json` 계약 (99_freeze·05_test가 읽는 방법)**
| 키 | 뜻 | 읽는 곳 |
|---|---|---|
| `valid` | `true`면 점검 통과 + 최종 결정. `false`면 쓰면 안 됨 | 99_freeze·05_test 모두 먼저 확인 |
| `selected_model` | `"xgb"` 또는 `"xgb_mono"` | 05_test: 고른 모델 |
| `a_star`, `ref_model` | 모형별 평균 곡선 a*, 공통 승인율 기준 모형 | 05_test: 공통 승인율 `a_star[ref_model]` (아키텍처 5단계) |
| `inputs` | 선택에 쓴 결과 파일 sha256 | 99_freeze: 지금 파일과 같은 실행인지 대조 |
| `delta`, `alpha`, `comparisons`, `split_hash`, `config_hash`, `lib_prep_hash`, `B`, `created` | 판정 근거 | 99_freeze 기록, 보고서 |

- **99_freeze (core)**: 아래 `verify_selection(OUT_DIR)`로 확인한 뒤(`valid`, 입력 지문 일치, 설정 해시 일치) `selection.json`을 FREEZE 폴더에 복사하고 sha256을 `FREEZE_RECORD.json`에 남깁니다. 실패하면 고정하지 않습니다.
- **05_test (eval)**: 실시간 `outputs/collect/`가 아니라 **FREEZE 폴더의 사본**을 읽고, sha256이 `FREEZE_RECORD.json`과 같은지 확인한 뒤 `selected_model`과 `a_star[ref_model]`을 씁니다. (지금 05_test는 `summary.json`의 회차 중앙값 a*를 읽으므로 바꿔야 함)
- 두 노트북은 다른 구역 파일이라 이 PR에서 고치지 않았습니다. 아래 함수를 그대로 붙여 쓰면 됩니다.

```python
def verify_selection(out_dir, config_hash=None, lib_prep_hash=None):
    """selection.json이 지금 결과 파일과 같은 실행에서 나온 최종 결정인지 확인. 반환: selection dict"""
    import json, hashlib
    from pathlib import Path
    out_dir = Path(out_dir); sel = json.loads((out_dir / "collect" / "selection.json").read_text(encoding="utf-8"))
    assert sel.get("valid") is True, f"selection.json 무효: {sel.get('reason')}"
    for rel, h in sel["inputs"].items():
        assert hashlib.sha256((out_dir / rel).read_bytes()).hexdigest() == h, f"{rel}가 선택 뒤에 바뀜 → 06_collect 다시 실행"
    if config_hash: assert sel["config_hash"] == config_hash, "config 해시 다름"
    if lib_prep_hash: assert sel["lib_prep_hash"] == lib_prep_hash, "lib_prep 해시 다름"
    return sel
```

In [ ]:
# 6. 짝지은 차이와 selection.json (공유 파일은 점검 통과 + 최종 결정일 때만 씀)
if FAILED or not SPLIT_OK:
    revoke_selection("점검 실패: 1절 FAIL 또는 분할 지문 불일치·누락 (%s)" % now_iso())
    raise RuntimeError("점검 실패(1절 FAIL 또는 분할 지문 불일치·누락) → 모델 선택을 계산하지 않습니다. 이전 selection.json은 무효화했습니다. "
                       "1절·1-1 출력의 비고를 담당자에게 전달해 고친 뒤 다시 실행하세요.")

def sr_at_astar(m):
    """모형 m의 회차별 SR을 자기 a*에서 꺼낸다. 반환: Series(index=b)"""
    cur = have[m]["curves"]
    s = cur[np.isclose(cur["rate"], A_STAR[m])].set_index("b")["sr"]
    assert s.index.is_unique and len(s) == have[m]["curves"]["b"].nunique(), f"{m}: a*에서 회차별 SR을 못 찾음"
    return s

def paired(a, b):
    """같은 회차끼리 a − b. 반환: (차이 Series, 요약 dict)"""
    d = (sr_at_astar(a) - sr_at_astar(b)).dropna()
    q = d.quantile([ALPHA, 0.5, 1 - ALPHA])
    return d, {"n": int(len(d)), "mean": float(d.mean()), "q_lo": float(q.iloc[0]), "median": float(q.iloc[1]),
               "q_hi": float(q.iloc[2]), "negative_rounds": int((d < 0).sum())}

DELTA_PENDING = ("DELTA" in PENDING) or (DELTA is None)
comp, rows = {}, []
if "xgb" in have and "logit" in have:
    _, comp["xgb_minus_logit"] = paired("xgb", "logit")
if "xgb" in have and "xgb_mono" in have and not DELTA_PENDING:
    _, comp["xgb_minus_mono"] = paired("xgb", "xgb_mono")
for k, label in (("xgb_minus_logit", "② − ① (ML 이득)"), ("xgb_minus_mono", "② − ③ (단조 제약 비용)")):
    if k in comp:
        c = comp[k]
        rows.append({"비교": label, "회차": c["n"], "평균": c["mean"], f"{ALPHA:.1%} 분위수": c["q_lo"],
                     "중앙값": c["median"], f"{1 - ALPHA:.1%} 분위수": c["q_hi"], "음수 회차": c["negative_rounds"]})
t4c = pd.DataFrame(rows).round(4)
t4c.to_csv(TAB_DIR / f"t4c_paired_selection{tag}.csv", index=False, encoding="utf-8-sig")
print(t4c.to_string(index=False) if len(t4c) else "짝지은 비교를 할 모형이 아직 없음")

ml_gain = comp["xgb_minus_logit"]["q_lo"] > 0 if "xgb_minus_logit" in comp else None
missing = [MODELS[m] for m in MODELS if m not in have]
if missing:
    selected, reason = None, "보류: 결과 없음 → " + ", ".join(missing)
elif DELTA_PENDING:
    selected, reason = None, "보류: δ 미정 (00_config DELTA 확정 뒤 다시 실행)"
elif comp["xgb_minus_mono"]["q_hi"] < DELTA:
    selected, reason = "xgb_mono", f"②−③ {1 - ALPHA:.1%} 분위수 {comp['xgb_minus_mono']['q_hi']:.4f} < δ {DELTA} → ③ 선택"
else:
    selected, reason = "xgb", f"②−③ {1 - ALPHA:.1%} 분위수 {comp['xgb_minus_mono']['q_hi']:.4f} ≥ δ {DELTA} → ② 선택"
print("\nML 이득(②−① 2.5% 분위수 > 0):", ml_gain, "| 선택:", selected, "|", reason)

if selected is None:
    revoke_selection(reason)
    print("최종 결정이 아니므로 새 selection.json은 쓰지 않았습니다.")
else:
    INPUTS = {k: h for m in MODELS for k, h in have[m]["inputs"].items()}   # 1절에서 파싱한 바로 그 바이트의 지문
    changed = [k for k, h in INPUTS.items() if read_hashed(OUT_DIR / k)[1] != h]
    if changed:
        revoke_selection("선택 계산 중 입력 파일이 바뀜: " + ", ".join(changed))
        raise RuntimeError(f"06_collect 실행 중에 결과 파일이 바뀜 {changed} → selection.json을 쓰지 않음. 업로드가 끝난 뒤 처음부터 다시 실행하세요.")
    selection = dict(
        valid=True, selected_model=selected, reason=reason, ml_gain=ml_gain,
        rule={"a_star": "평균 곡선 최대 (MIN_RATE 이상)", "ml_gain": f"②−① {ALPHA:.1%} 분위수 > 0",
              "select_mono": f"②−③ {1 - ALPHA:.1%} 분위수 < δ"},
        a_star={m: A_STAR[m] for m in A_STAR}, ref_model=REF_MODEL, comparisons=comp, delta=DELTA, alpha=ALPHA,
        min_rate=MIN_RATE, B={m: int(have[m]["res"].shape[0]) for m in have},
        split_hash=next(iter(split.values())), config_hash=NOW["config_hash"], lib_prep_hash=NOW["lib_prep_hash"],
        inputs=INPUTS, created=now_iso())
    SEL_DIR.mkdir(parents=True, exist_ok=True)
    SEL_PATH.write_text(json.dumps(selection, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    print("저장:", SEL_PATH.relative_to(ROOT),
          "| sha256", hashlib.sha256(SEL_PATH.read_bytes()).hexdigest()[:12], "(99_freeze 기록과 대조)")